# Session 2 — GPU: Sequence Length Scaling

## Background

Attention's memory footprint grows as O(seq²): the attention matrix is `[batch, heads, seq_len, seq_len]` and must be materialized and transposed at every step. Session 1 showed that the GPU's memory bus (~300 GB/s) becomes the bottleneck as batch size grows. Sequence length applies the same pressure through a different dimension, but with a quadratic rather than linear relationship. The TPU's HBM2 bus (~819 GB/s) provides 2.7× more bandwidth — if this is the binding constraint, the gap should widen as sequences grow.

## Goal

At the end of this session, participants will be able to explain how attention's quadratic memory footprint interacts differently with each device's memory architecture, will have measured how the throughput ratio shifts as sequences lengthen, and will know which device to reach for when their research involves long-context models.

## Question

Does the TPU's memory bandwidth advantage compound as sequences grow, or does the crossover point move?

---

**Runtime:** GPU (NVIDIA L4 or equivalent)

After running, results are saved to `results/gpu_seq_sweep.json` and loaded automatically by `03-analysis.ipynb`.

In [ ]:
import time, torch, torch.nn as nn

assert torch.cuda.is_available(), "No GPU found. Runtime → Change runtime type → GPU."

device = torch.device("cuda")
print(f"Device  : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

In [ ]:
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 2 config ──────────────────────────────────────────────────────────
BATCH_SIZE  = 32   # fixed at Session 1 crossover point
SEQ_LENGTHS = [64, 128, 256, 512, 1024, 2048]
N_STEPS, WARMUP = 50, 5

# ── Benchmark function ────────────────────────────────────────────────────────
def benchmark(seq_len, device):
    model     = BenchmarkTransformerBlock(d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()
    model.train()
    elapsed = 0.0
    for step in range(N_STEPS + WARMUP):
        x = torch.randn(BATCH_SIZE, seq_len, D_MODEL, device=device)
        y = torch.randn(BATCH_SIZE, seq_len, D_MODEL, device=device)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        if step >= WARMUP:
            elapsed += t1 - t0
    return (N_STEPS * BATCH_SIZE) / elapsed

print(f"Config  : batch={BATCH_SIZE}, d_model={D_MODEL}, n_head={N_HEAD}, dim_ff={DIM_FEEDFORWARD}")
print(f"Sweep   : seq_len = {SEQ_LENGTHS}")
print("Model and benchmark function defined.")

In [ ]:
results = {}

for seq_len in SEQ_LENGTHS:
    try:
        print(f"  [GPU] seq_len={seq_len:>5} ... ", end="", flush=True)
        tput = benchmark(seq_len, device)
        results[seq_len] = round(tput, 1)
        vram = torch.cuda.memory_allocated() / 1e9
        print(f"{tput:,.0f} samples/sec  (VRAM: {vram:.2f} GB)")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("OOM — stopping.")
            torch.cuda.empty_cache()
            break
        raise

print("\nDone.")
print("\n── Copy this into 03-analysis.ipynb ──────────────────")
print(f"GPU_RESULTS = {results}")

In [ ]:
import json, os
from datetime import datetime, timezone

def save_results(hardware, results, device_name="", fixed_batch_size=None, results_dir="results"):
    os.makedirs(results_dir, exist_ok=True)
    payload = {
        "hardware":         hardware,
        "device_name":      device_name,
        "session":          "2",
        "axis":             "seq_len",
        "fixed_batch_size": fixed_batch_size,
        "timestamp":        datetime.now(timezone.utc).isoformat(),
        "results":          {str(k): v for k, v in sorted(results.items())},
    }
    path = os.path.join(results_dir, f"{hardware.lower()}_seq_sweep.json")
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    return path

path = save_results(
    "GPU", results,
    device_name=torch.cuda.get_device_name(0),
    fixed_batch_size=BATCH_SIZE,
    results_dir="results",
)
print(f"Saved → {path}")